# 🧹 Step 2: Data Cleaning & Feature Engineering (`02_data_cleaning.ipynb`)

### Overview of Data Cleaning Steps
1. **Load Raw Data:** Reads `data/earthquakes_intermediate.csv` (`104,464` rows) and renames `depth` $\rightarrow$ `depth_km`.
2. **Parse Dates:** Uses `pd.to_datetime()` on `time` and `updated`.
3. **Extract & Clean Country:** Uses a custom function `def get_country(place):` with `.apply(get_country)` and `.replace(state_map)`.
4. **Clean String Columns:** Uses `.str.lower()` and `.str.strip()`.
5. **Treat Missing Values:**
   - Continuous columns (`depth_km`, `nst`, `dmin`, `rms`, `gap`, `magError`, `depthError`, `magNst`, `sig`) are **Right Skewed (`mean > median`)**, so we fill missing values with the **Median**.
   - Categorical `alert` is filled using **Domain-Based Imputation** (`"unassessed"`).
6. **Feature Engineering:** Extracts `Year`, `Month`, `Day`, `Hour`, `Day_of_week`, `depth_flag` (`shallow` vs `deep`), and `severity_flag` (`destructive` vs `standard`).

In [1]:
import os
import re
import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

INPUT_PATH = "data/earthquakes_intermediate.csv"
OUTPUT_PATH = "data/earthquakes_cleaned.csv"

df = pd.read_csv(INPUT_PATH)
print("Initial shape (rows, columns):", df.shape)
print("\nMissing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Initial shape (rows, columns): (104464, 29)

Missing values before cleaning:
place             492
nst              6728
dmin             7155
gap              6729
magError         7341
depthError          2
magNst           7330
alert          100209
dtype: int64


### 2.1 Rename `depth` Column & Convert Timestamps (`pd.to_datetime`)

In [2]:
# Rename 'depth' to 'depth_km'
df.rename(columns={"depth": "depth_km"}, inplace=True)

# Convert 'time' and 'updated' to datetime
df["time"] = pd.to_datetime(df["time"], utc=True, errors="coerce").dt.tz_localize(None)
df["updated"] = pd.to_datetime(df["updated"], utc=True, errors="coerce").dt.tz_localize(None)

print(df[["time", "updated", "depth_km"]].head(3))

time                  updated  depth_km
0  2021-01-31 23:20:49.923  2021-04-16 19:02:44.040     17.27
1  2021-01-31 23:08:17.161  2021-04-16 19:03:47.040    426.71
2  2021-01-31 22:54:19.760  2021-04-16 19:03:47.040     46.73


### 2.2 Extract & Standardize `country` Using `.apply()` and `.replace()`

In [3]:
def get_country(place):
    if pd.isnull(place):
        return "Unknown"
    parts = re.split(r",\s*", str(place))
    country = parts[-1].strip()
    country = re.sub(r"\s+region$", "", country, flags=re.IGNORECASE).strip()
    return country if country != "" else "Unknown"

df["country"] = df["place"].apply(get_country)

state_map = {
    "CA": "California",
    "AK": "Alaska",
    "HI": "Hawaii",
    "PR": "Puerto Rico",
    "NV": "Nevada",
    "TX": "Texas",
    "OK": "Oklahoma",
    "ID": "Idaho",
    "UT": "Utah",
    "MT": "Montana",
    "WA": "Washington",
    "OR": "Oregon",
    "WY": "Wyoming",
    "NM": "New Mexico",
    "AZ": "Arizona",
    "CO": "Colorado",
    "KS": "Kansas",
    "AR": "Arkansas",
    "TN": "Tennessee",
    "MO": "Missouri",
    "SC": "South Carolina",
    "NC": "North Carolina",
    "VA": "Virginia",
    "NY": "New York",
    "ME": "Maine",
    "MX": "Mexico",
    "B.C.": "Canada",
}
df["country"] = df["country"].replace(state_map)
print("Total unique countries/regions:", df["country"].nunique())
print(df["country"].value_counts().head(10))

Total unique countries/regions: 493
country
Alaska                    12425
Indonesia                  7840
Japan                      6292
Russia                     5712
Philippines                4351
Papua New Guinea           3397
Puerto Rico                3152
South Sandwich Islands     3025
Tonga                      2890
Chile                      2865
Name: count, dtype: int64


### 2.3 Clean String Columns & Treat Missing Values (Median & Domain Fill)

In [4]:
# Clean string columns using .str.lower() and .str.strip()
string_cols = ["magType", "type", "status", "net", "locationSource", "magSource", "types", "ids", "sources"]
for col in string_cols:
    df[col] = df[col].fillna("unknown").astype(str).str.strip().str.lower()

# Domain-based imputation for 'alert' and 'place'
df["alert"] = df["alert"].fillna("unassessed").astype(str).str.strip().str.lower()
df["place"] = df["place"].fillna("Unknown").astype(str).str.strip()

# Numeric columns: Right Skewed (mean > median), so impute with Median
num_cols = [
    "latitude", "longitude", "depth_km", "mag", "nst",
    "dmin", "rms", "gap", "magError", "depthError",
    "magNst", "sig", "tsunami",
]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

median_cols = ["depth_km", "mag", "nst", "dmin", "rms", "gap", "magError", "depthError", "magNst", "sig"]
for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

df["tsunami"] = df["tsunami"].fillna(0).astype(int)
df["sig"] = df["sig"].astype(int)
df["nst"] = df["nst"].astype(int)
df["magNst"] = df["magNst"].astype(int)

print("Remaining nulls:", df.isnull().sum().sum())

Remaining nulls: 0


### 2.4 Remove Duplicates & Create Derived Analytical Columns

In [5]:
# Remove duplicate rows by 'id' and reset index
df.drop_duplicates(subset="id", keep="first", inplace=True)
df.reset_index(drop=True, inplace=True)

# Extract Date/Time features using .dt accessor
df["Year"] = df["time"].dt.year
df["Month"] = df["time"].dt.month
df["Day"] = df["time"].dt.day
df["Hour"] = df["time"].dt.hour
df["Day_of_week"] = df["time"].dt.day_name()

# Create binary flags: depth_flag and severity_flag
df["depth_flag"] = np.where(df["depth_km"] <= 70, "shallow", "deep")
df["severity_flag"] = np.where(df["mag"] >= 6.0, "destructive", "standard")

df.to_csv(OUTPUT_PATH, index=False)
print("Saved cleaned dataset to:", OUTPUT_PATH)
print("Final Cleaned Shape:", df.shape)
df[["time", "country", "mag", "depth_km", "Year", "Day_of_week", "depth_flag", "severity_flag"]].head()

Saved cleaned dataset to: data/earthquakes_cleaned.csv
Final Cleaned Shape: (104464, 35)
                      time  ... severity_flag
0  2021-01-31 23:20:49.923  ...      standard
1  2021-01-31 23:08:17.161  ...      standard
2  2021-01-31 22:54:19.760  ...      standard
3  2021-01-31 22:06:00.832  ...      standard
4  2021-01-31 21:51:14.016  ...      standard

[5 rows x 8 columns]
